In [42]:
import pandas as pd
import numpy as np
import os
from functools import reduce
from scipy import stats
import cvxpy as cp
from datetime import datetime
from collections import defaultdict
import pickle
import lightgbm as lgb
from scipy.stats import pearsonr, spearmanr

srcdir = "E:/SJTU/实习/国泰海通/barra因子/data_base"
adir = "E:/SJTU/实习/国泰海通/barra因子/result"
model_dir = "E:/SJTU/实习/国泰海通/barra因子/data_base/lgb_models"


def orthogonalize_pair(df, col1, col2, standardize=False):
    x1 = df[col1].values
    x2 = df[col2].values
    # 去均值（等价于带截距回归）
    x1_mean = x1.mean()
    x2_mean = x2.mean()
    x1_c = x1 - x1_mean
    x2_c = x2 - x2_mean
    # OLS beta
    beta1 = np.dot(x1_c, x2_c) / np.dot(x1_c, x1_c)
    beta0 = x2_mean - beta1 * x1_mean
    # 残差（正交部分）
    x2_res = x2 - beta0 - beta1 * x1
    df.loc[:, col2] = x2_res
    return df

params = {
        "objective":'regression',
        "boosting_type":"gbdt",
        # 树参数
        "n_estimators":300, 
        "learning_rate":0.03,
        "num_leaves":10,
        "max_depth":4,
        # 稳定性
        "random_state":42,
        "n_jobs":16,
        "force_row_wise":True
}
start_dt = "2024-09-01"
end_dt = "2026-03-25"
trdates = pd.read_pickle(f"{srcdir}/trading_dates.pkl")
tr_filter = [d for d in trdates if (d >= start_dt) and (d <= end_dt)]
window = 60
purge = 5

mcp_dict = pd.read_pickle(f"{srcdir}/stk_mcp/全A_freemcp_25_26D_dict.pkl")
return_dict = pd.read_pickle(f"{srcdir}/stk_ret/全A_ret_24_2603D_1030minute_s-1_dict.pkl")
alpha_dict = pd.read_pickle(f"{adir}/延迟alpha/ortho_delay_measures_2024_2026_dict.pkl")
alpha_name = "D1"#"DELAY"#'MACD_HIST'


for dt in tr_filter[window+purge:]:
    # if w0 is None:
    idx = window+purge
    traing_dates = tr_filter[idx-window-purge:idx-purge]
    res = []
    for tr_dt in traing_dates:
        tr_dt_tp = pd.to_datetime(tr_dt)
        df_alpha = alpha_dict[tr_dt_tp].reset_index()
        df_mcp = mcp_dict[tr_dt_tp][["free_circulation"]]

        tr_dt_tp = tr_dt_tp.replace(hour=10, minute=30, second=0)
        df_ret_tz = return_dict[tr_dt_tp].to_frame()
        df_ret_tz.columns = ["ret_tz"]

        dfs = [df_alpha,df_mcp,df_ret_tz]
        df_t = reduce(lambda left, right: pd.merge(left, right,on="order_book_id",how="inner"), dfs)
        df_t.reset_index(inplace=True)
        df_t["date"] = tr_dt
        res.append(df_t.set_index(['order_book_id', 'date']))
    df_training = pd.concat(res,ignore_index=False)
    df_training.dropna(how="any",inplace=True)

    X_train = df_training[[alpha_name]]
    y_train = df_training["ret_tz"]
    sample_weight = np.sqrt(df_training["free_circulation"])
    # 防止极端值
    sample_weight = sample_weight / sample_weight.median()
    sample_weight = sample_weight.clip(lower=0.1,upper=10)

    dataset = lgb.Dataset(X_train, label=y_train,weight=sample_weight)
    model = lgb.train(params, dataset)

    #检查in-sample performance
    pred_train = model.predict(X_train)
    pearson_ic = pearsonr(pred_train,y_train)[0]
    rank_ic = spearmanr(pred_train,y_train)[0]

    ss_res = np.sum((y_train - pred_train) ** 2)
    ss_tot = np.sum((y_train - y_train.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    print(f"Train Pearson IC : {pearson_ic:.6f}")
    print(f"Train Rank IC    : {rank_ic:.6f}")
    print(f"Train R^2        : {r2:.6f}")
    #model_dict[dt] = model


    model_path = f"{model_dir}/{dt}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"saved model: {dt}")

    break

    


[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 292038, number of used features: 1
[LightGBM] [Info] Start training from score 0.006705
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [43]:
df_eval = pd.DataFrame({

    "pred": pred_train,
    "ret": y_train.values

})

df_eval["group"] = pd.qcut(
    df_eval["pred"],
    10,
    labels=False,
    duplicates='drop'
)

group_ret = (
    df_eval
    .groupby("group")["ret"]
    .mean()
)

# long-short
ls_ret = (
    group_ret.iloc[-1]
    - group_ret.iloc[0]
)

print(f"Train Long-Short : {ls_ret:.6f}")

print("\nGroup Return:")

print(group_ret)

print("=" * 60)

Train Long-Short : 0.004442

Group Return:
group
0    0.004769
1    0.005923
2    0.006315
3    0.006604
4    0.007147
5    0.007099
6    0.007143
7    0.007760
8    0.007902
9    0.009210
Name: ret, dtype: float64


In [41]:
y_train

order_book_id  date      
000001.XSHE    2024-09-02   -0.013752
000002.XSHE    2024-09-02   -0.007610
000004.XSHE    2024-09-02    0.008554
000006.XSHE    2024-09-02    0.080311
000007.XSHE    2024-09-02   -0.012987
                               ...   
688799.XSHG    2024-12-03    0.000983
688800.XSHG    2024-12-03    0.025689
688819.XSHG    2024-12-03   -0.011458
688981.XSHG    2024-12-03    0.021747
689009.XSHG    2024-12-03   -0.024594
Name: ret_tz, Length: 302110, dtype: float64